# Final Experiment: 3x2 Known-Task Monitoring
This notebook runs the full transcript-structure (3) x processing-mode (2) experiment with repeated resumable runs.

## 1. Configuration

In [1]:
from pathlib import Path
import sys

repo = Path.cwd().resolve().parent if Path.cwd().name == 'code' else Path.cwd().resolve()
sys.path.insert(0, str(repo / 'code' / 'src'))

from swisstext_eval.config import (
    DebugConfig,
    EvalConfig,
    ExperimentConfig,
    NotebookConfig,
    PathsConfig,
    PromptConfig,
    ProviderConfig,
)

config = NotebookConfig(
    paths=PathsConfig(
        repo_root=repo,
        input_data_dir=repo / 'data' / 'scenarios' / 'synthetic_firefighter_radio_controlled_v1',
        output_root=repo / 'code' / 'outputs',
        cache_dir=repo / 'code' / '.cache',
    ),
    provider=ProviderConfig(mode='live', model_name='gpt-5.2', temperature=0.0),
    experiment=ExperimentConfig(
        run_count=12,
        selected_scenarios=[],
        selected_structure_conditions=['structured_dialogue', 'no_speaker', 'continuous_transcript'],
        selected_processing_modes=['full_transcript', 'incremental'],
        incremental_prefix_policy='every_message',
        api_batch_size=8,
        resume_behavior='skip_completed',
        output_run_tag='final_eval_20260321',
        prompt_version_tag='v3_known_task_monitoring',
    ),
    prompts=PromptConfig(
        system_prompt_path=repo / 'code' / 'prompts' / 'system_prompt.txt',
        user_prompt_template_path=repo / 'code' / 'prompts' / 'user_prompt_template.txt',
    ),
    evaluation=EvalConfig(
        export_tables=True,
        export_figures=True,
        export_prefix_level_metrics=True,
        export_detailed_condition_rows=True,
    ),
    debug=DebugConfig(verbose=True, limit_scenarios=None, limit_work_items=None),
)
config

NotebookConfig(paths=PathsConfig(repo_root=WindowsPath('C:/Data/VSCodeWorkSpace/swisstext26'), input_data_dir=WindowsPath('C:/Data/VSCodeWorkSpace/swisstext26/data/scenarios/synthetic_firefighter_radio_controlled_v1'), output_root=WindowsPath('C:/Data/VSCodeWorkSpace/swisstext26/code/outputs'), cache_dir=WindowsPath('C:/Data/VSCodeWorkSpace/swisstext26/code/.cache')), provider=ProviderConfig(model_name='gpt-5.2', api_key_env='OPENAI_API_KEY', temperature=0.0, max_tokens=1200, timeout_seconds=60, retry_count=2, mode='live'), experiment=ExperimentConfig(run_count=12, selected_scenarios=[], selected_structure_conditions=['structured_dialogue', 'no_speaker', 'continuous_transcript'], selected_processing_modes=['full_transcript', 'incremental'], incremental_prefix_policy='every_message', api_batch_size=8, resume_behavior='skip_completed', output_run_tag='final_eval_20260321', prompt_version_tag='v3_known_task_monitoring'), prompts=PromptConfig(system_prompt_path=WindowsPath('C:/Data/VSCodeW

## 2. Imports and Environment Setup

In [2]:
import pandas as pd
from swisstext_eval.io.scenarios import load_scenarios

## 3. Scenario Loading and Validation

In [3]:
scenarios = load_scenarios(config.paths.input_data_dir, config.experiment.selected_scenarios)
[(s.scenario_id, len(s.messages), len(s.predefined_tasks)) for s in scenarios]

[('S1', 19, 3), ('S2', 21, 3), ('S3', 16, 3), ('S4', 24, 3), ('S5', 22, 3)]

## 4. Transcript Condition Generation

In [4]:
from swisstext_eval.transforms.transcript_conditions import render_transcript
sample = scenarios[0]
{c: render_transcript(sample, c)[:220] for c in config.experiment.selected_structure_conditions}

{'structured_dialogue': 'Einsatzleitung: An Angriffstrupp und Wassertrupp von Einsatzleitung, Meldung: Kuechenbrand im 2. Obergeschoss links, Verrauchung im Treppenhaus, Personenlage unklar, Angriffstrupp und Wassertrupp antworten\nAngriffstrupp:',
 'no_speaker': 'An Angriffstrupp und Wassertrupp von Einsatzleitung, Meldung: Kuechenbrand im 2. Obergeschoss links, Verrauchung im Treppenhaus, Personenlage unklar, Angriffstrupp und Wassertrupp antworten\nAngriffstrupp verstanden, ruec',
 'continuous_transcript': 'An Angriffstrupp und Wassertrupp von Einsatzleitung, Meldung: Kuechenbrand im 2. Obergeschoss links, Verrauchung im Treppenhaus, Personenlage unklar, Angriffstrupp und Wassertrupp antworten Angriffstrupp verstanden, ruec'}

## 5. Prompt Building

In [5]:
from swisstext_eval.prompts.builders import load_prompt
load_prompt(config.prompts.system_prompt_path)[:300]

'\ufeffYou are monitoring predefined operational firefighter tasks.\n\nYour job is not to invent tasks.\nOnly assess the state of the provided task list from transcript evidence.\n\nRules:\n- Be conservative and evidence-based.\n- Do not hallucinate assignments, units, completions, or outcomes.\n- If evidence is '

## 6. Request Execution / Resume-Aware Runner

In [6]:
from tqdm.auto import tqdm
from swisstext_eval.eval.repeated import count_total_work_items, run_repeated_evaluation

total_items = count_total_work_items(config)
progress = tqdm(total=total_items, desc='Experiment progress', unit='work_item')

def _on_progress(current, total, meta):
    progress.n = current
    prefix = 'full' if meta['prefix_index'] == -1 else meta['prefix_index']
    progress.set_postfix_str(
        f"run={meta['run_index']}/{meta['run_count']} | {meta['scenario_id']} | "
        f"{meta['structure_condition']} | {meta['processing_mode']} | p={prefix}"
    )
    progress.refresh()

batch_result = run_repeated_evaluation(config, progress_callback=_on_progress)
progress.n = total_items
progress.close()
batch_result


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Experiment progress: 100%|██████████| 3852/3852 [2:11:44<00:00,  2.05s/work_item, run=12/12 | S5 | continuous_transcript | incremental | p=22]       


{'batch_id': 'final_eval_20260321__batch',
 'batch_dir': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch',
 'run_count': 12,
 'manifest': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\batch_manifest.json',
 'all_condition_rows_csv': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\all_condition_rows.csv',
 'full_transcript_summary_csv': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\full_transcript_summary.csv',
 'incremental_summary_csv': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\incremental_summary.csv',
 'combined_condition_summary_csv': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\combined_condition_summary.csv',
 'per_scenario_summary_csv': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tabl

## 7. Parsing

In [7]:
# Parsed outputs are persisted under each run folder: parsed_predictions/ and requests/.
batch_result['manifest']

'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\batch_manifest.json'

## 8. Metric Computation

In [8]:
full_summary = pd.read_csv(batch_result['full_transcript_summary_csv'])
inc_summary = pd.read_csv(batch_result['incremental_summary_csv'])
terminal_convergence = pd.read_csv(batch_result['terminal_convergence_summary_csv'])

paper_latency = inc_summary[[
    'structure_condition',
    'assignment_detection_latency_mean',
    'assignment_detection_latency_ci95',
    'assignment_detection_miss_rate_mean',
    'assignment_detection_miss_rate_ci95',
    'completion_detection_latency_mean',
    'completion_detection_latency_ci95',
    'completion_detection_miss_rate_mean',
    'completion_detection_miss_rate_ci95',
]]

full_summary, inc_summary, paper_latency, terminal_convergence

(     structure_condition  n_runs  assignment_accuracy_mean  \
 0  continuous_transcript      12                       1.0   
 1             no_speaker      12                       1.0   
 2    structured_dialogue      12                       1.0   
 
    assignment_accuracy_std  assignment_accuracy_ci95  \
 0                      0.0                       0.0   
 1                      0.0                       0.0   
 2                      0.0                       0.0   
 
    unit_assignment_accuracy_mean  unit_assignment_accuracy_std  \
 0                            1.0                           0.0   
 1                            1.0                           0.0   
 2                            1.0                           0.0   
 
    unit_assignment_accuracy_ci95  completion_accuracy_mean  \
 0                            0.0                  0.933333   
 1                            0.0                  0.933333   
 2                            0.0                  0.9333

## 9. Aggregation

In [9]:
combined = pd.read_csv(batch_result['combined_condition_summary_csv'])
combined.head(20)

,structure_condition,processing_mode,metric_name,mean,std,ci95
0,continuous_transcript,full_transcript,assignment_accuracy,1.000000,0.000000,0.000000
1,continuous_transcript,full_transcript,assignment_detection_latency,NaN,0.000000,0.000000
2,continuous_transcript,full_transcript,assignment_detection_miss_rate,NaN,0.000000,0.000000
3,continuous_transcript,full_transcript,completion_accuracy,0.933333,0.134459,0.034023
4,continuous_transcript,full_transcript,completion_detection_latency,NaN,0.000000,0.000000
5,continuous_transcript,full_transcript,completion_detection_miss_rate,NaN,0.000000,0.000000
6,continuous_transcript,full_transcript,completion_outcome_accuracy,0.000000,0.000000,0.000000
7,continuous_transcript,full_transcript,current_state_accuracy,NaN,0.000000,0.000000
8,continuous_transcript,full_transcript,final_state_accuracy,0.933333,0.134459,0.034023
9,continuous_transcript,full_transcript,unit_assignment_accuracy,1.000000,0.000000,0.000000


## 10. Table Export

In [10]:
batch_result['all_condition_rows_csv'], batch_result['full_transcript_summary_csv'], batch_result['incremental_summary_csv'], batch_result['terminal_convergence_per_scenario_csv'], batch_result['terminal_convergence_summary_csv']

('C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\all_condition_rows.csv',
 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\full_transcript_summary.csv',
 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\incremental_summary.csv',
 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\terminal_convergence_per_scenario.csv',
 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\terminal_convergence_summary.csv')

## 11. Figure Export

In [11]:
{k: v for k, v in batch_result.items() if k.startswith('figure_')}

{'figure_full_transcript': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\figures\\full_transcript_metrics_by_structure.png',
 'figure_incremental': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\figures\\incremental_metrics_by_structure.png',
 'figure_latency': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\figures\\incremental_latency_by_structure.png',
 'figure_structure_processing_comparison': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\figures\\structure_processing_comparison.png'}

## 12. Final Run Summary

In [12]:
summary = {
    'batch_id': batch_result['batch_id'],
    'run_count': batch_result['run_count'],
    'batch_dir': batch_result['batch_dir'],
    'incremental_summary_csv': batch_result['incremental_summary_csv'],
    'terminal_convergence_summary_csv': batch_result['terminal_convergence_summary_csv'],
}
summary

{'batch_id': 'final_eval_20260321__batch',
 'run_count': 12,
 'batch_dir': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch',
 'incremental_summary_csv': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\incremental_summary.csv',
 'terminal_convergence_summary_csv': 'C:\\Data\\VSCodeWorkSpace\\swisstext26\\code\\outputs\\final_eval_20260321__batch\\tables\\terminal_convergence_summary.csv'}